# TagoreGPT: Rabindra Sangeet guide

This notebook builds a resumable SQLite catalogue from the song pages listed in Geetabitan's sitemap, answers metadata questions from that database, and trains a small character-level transformer on the site's Romanized lyric text.

The fact answers come from the database, with the source page attached. The tiny transformer is a text-continuation experiment adapted from the supplied gpt_dev.py; it is not the fact store and should not be trusted to recall taal, parjaay, raag, or ritu from memory.

The site currently reports 2,179 catalogue entries; its sitemap has one extra lyric-page URL. This notebook deduplicates exact URLs and reports the count it actually sees. It follows robots.txt, pauses between page requests, saves progress in SQLite, and resumes after interruption.

**Known source limitation:** individual pages show Bengali lyrics as images in many cases. This first pass saves the image URL and the plain-text transliteration. Bengali OCR is left out so unreviewed OCR errors do not enter the training corpus. Missing ritu/season values remain blank rather than being guessed.

In [ ]:
%pip -q install requests beautifulsoup4

In [ ]:
import json
import re
import sqlite3
import time
import urllib.robotparser
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from difflib import SequenceMatcher
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup, NavigableString
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://www.geetabitan.com"
ROBOTS_URL = BASE_URL + "/robots.txt"
USER_AGENT = "TagoreGPT educational research/0.1"
DB_PATH = "tagore_songs.sqlite"
REQUEST_DELAY_SECONDS = 0.4

session = requests.Session()
retry = Retry(
    total=3,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)
session.mount("https://", HTTPAdapter(max_retries=retry))
session.headers.update({"User-Agent": USER_AGENT})

robots_response = session.get(ROBOTS_URL, timeout=30)
robots_response.raise_for_status()
robots = urllib.robotparser.RobotFileParser()
robots.set_url(ROBOTS_URL)
robots.parse(robots_response.text.splitlines())

if not robots.can_fetch(USER_AGENT, BASE_URL + "/lyrics/"):
    raise RuntimeError("robots.txt does not allow this crawl path.")

sitemap_match = re.search(r"(?im)^Sitemap:\s*(\S+)", robots_response.text)
SITEMAP_URL = sitemap_match.group(1) if sitemap_match else "https://geetabitan.com/sitemap.xml"
sitemap_response = session.get(SITEMAP_URL, timeout=60)
sitemap_response.raise_for_status()
root = ET.fromstring(sitemap_response.content)
all_urls = [
    (node.text or "").strip()
    for node in root.iter()
    if node.tag.lower().endswith("loc") and node.text
]

song_urls = sorted({
    url for url in all_urls
    if urlparse(url).path.lower().startswith("/lyrics/")
    and urlparse(url).path.lower().endswith("-lyric.html")
})

print(f"Sitemap: {SITEMAP_URL}")
print(f"Unique lyric-page URLs found: {len(song_urls):,}")
print("The catalogue page reports 2,179 entries; the sitemap may contain a duplicate or extra page.")
print("First URL:", song_urls[0] if song_urls else "none")

In [ ]:
FIELD_ALIASES = {
    "parjaay": "parjaay",
    "parjay": "parjaay",
    "taal": "taal",
    "tal": "taal",
    "raag": "raag",
    "raga": "raag",
    "ritu": "ritu",
    "rhitu": "ritu",
    "season": "ritu",
    "written on": "written_on",
    "written": "written_on",
    "year": "written_on",
    "collection": "collection",
    "swarabitan": "swarabitan",
    "notation by": "notation_by",
}


def clean_text(node):
    if node is None:
        return ""
    return "\n".join(
        line.strip()
        for line in node.get_text("\n", strip=True).splitlines()
        if line.strip()
    )


def text_after_heading(soup, phrase):
    phrase = phrase.lower()
    for heading in soup.find_all(["h2", "h3"]):
        if phrase in clean_text(heading).lower():
            pre = heading.find_next("pre")
            return clean_text(pre) if pre else None
    return None


def parse_song_page(html, source_url):
    soup = BeautifulSoup(html, "html.parser")

    welcome = soup.select_one(".welcome")
    title_roman = None
    if welcome:
        direct_text = "".join(
            str(child) for child in welcome.contents
            if isinstance(child, NavigableString)
        ).strip()
        title_roman = re.sub(r"\s+", " ", direct_text) or None

    metadata = {}
    for paragraph in soup.select("div.marginbottom20.fontengsize16 p"):
        line = clean_text(paragraph)
        match = re.match(r"^([^:]{1,50})\s*:\s*(.*)$", line)
        if not match:
            continue
        raw_label, value = match.group(1).strip().lower(), match.group(2).strip()
        key = FIELD_ALIASES.get(raw_label)
        if key:
            metadata[key] = value or None
        elif value:
            metadata[raw_label] = value

    lyrics_image_url = None
    for heading in soup.find_all(["h2", "h3"]):
        if "lyric in bengali" in clean_text(heading).lower():
            image = heading.find_next("img", src=True)
            if image:
                lyrics_image_url = urljoin(source_url, image["src"])
            break

    transliteration = text_after_heading(soup, "transliteration in english")
    translation = (
        text_after_heading(soup, "english translation")
        or text_after_heading(soup, "translation in english")
    )

    known_fields = set(FIELD_ALIASES.values())
    extra_metadata = {
        key: value for key, value in metadata.items()
        if key not in known_fields
    }

    return {
        "source_url": source_url,
        "title_roman": title_roman,
        "title_bengali": None,
        "lyrics_bengali": None,
        "lyrics_image_url": lyrics_image_url,
        "transliteration": transliteration,
        "translation": translation,
        "parjaay": metadata.get("parjaay"),
        "taal": metadata.get("taal"),
        "raag": metadata.get("raag"),
        "ritu": metadata.get("ritu"),
        "written_on": metadata.get("written_on"),
        "collection": metadata.get("collection"),
        "swarabitan": metadata.get("swarabitan"),
        "notation_by": metadata.get("notation_by"),
        "extra_metadata_json": json.dumps(extra_metadata, ensure_ascii=False),
        "scraped_at": datetime.now(timezone.utc).isoformat(),
    }


CREATE_SQL = '''
CREATE TABLE IF NOT EXISTS songs (
    source_url TEXT PRIMARY KEY,
    title_roman TEXT,
    title_bengali TEXT,
    lyrics_bengali TEXT,
    lyrics_image_url TEXT,
    transliteration TEXT,
    translation TEXT,
    parjaay TEXT,
    taal TEXT,
    raag TEXT,
    ritu TEXT,
    written_on TEXT,
    collection TEXT,
    swarabitan TEXT,
    notation_by TEXT,
    extra_metadata_json TEXT,
    scraped_at TEXT NOT NULL,
    scrape_status TEXT NOT NULL
);
CREATE INDEX IF NOT EXISTS idx_songs_title ON songs(title_roman);
CREATE INDEX IF NOT EXISTS idx_songs_taal ON songs(taal);
CREATE INDEX IF NOT EXISTS idx_songs_parjaay ON songs(parjaay);
CREATE INDEX IF NOT EXISTS idx_songs_raag ON songs(raag);
CREATE INDEX IF NOT EXISTS idx_songs_ritu ON songs(ritu);

CREATE TABLE IF NOT EXISTS scrape_log (
    source_url TEXT PRIMARY KEY,
    status TEXT NOT NULL,
    http_status INTEGER,
    error TEXT,
    attempted_at TEXT NOT NULL
);
'''

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
conn.executescript(CREATE_SQL)
conn.commit()
print("Database schema ready:", DB_PATH)

In [ ]:
INSERT_SQL = '''
INSERT INTO songs (
    source_url, title_roman, title_bengali, lyrics_bengali, lyrics_image_url,
    transliteration, translation, parjaay, taal, raag, ritu, written_on,
    collection, swarabitan, notation_by, extra_metadata_json, scraped_at, scrape_status
) VALUES (
    :source_url, :title_roman, :title_bengali, :lyrics_bengali, :lyrics_image_url,
    :transliteration, :translation, :parjaay, :taal, :raag, :ritu, :written_on,
    :collection, :swarabitan, :notation_by, :extra_metadata_json, :scraped_at, 'ok'
)
ON CONFLICT(source_url) DO UPDATE SET
    title_roman=excluded.title_roman,
    title_bengali=excluded.title_bengali,
    lyrics_bengali=excluded.lyrics_bengali,
    lyrics_image_url=excluded.lyrics_image_url,
    transliteration=excluded.transliteration,
    translation=excluded.translation,
    parjaay=excluded.parjaay,
    taal=excluded.taal,
    raag=excluded.raag,
    ritu=excluded.ritu,
    written_on=excluded.written_on,
    collection=excluded.collection,
    swarabitan=excluded.swarabitan,
    notation_by=excluded.notation_by,
    extra_metadata_json=excluded.extra_metadata_json,
    scraped_at=excluded.scraped_at,
    scrape_status='ok'
'''


done = {
    row["source_url"]
    for row in conn.execute("SELECT source_url FROM songs WHERE scrape_status='ok'")
}
pending_urls = [url for url in song_urls if url not in done]
print(f"Already saved: {len(done):,}; remaining: {len(pending_urls):,}")
print("This is a polite sequential crawl and can take several minutes. Progress commits every 20 pages.")

for index, source_url in enumerate(pending_urls, start=1):
    attempted_at = datetime.now(timezone.utc).isoformat()
    http_status = None
    try:
        response = session.get(source_url, timeout=30)
        http_status = response.status_code
        response.raise_for_status()
        record = parse_song_page(response.text, source_url)
        conn.execute(INSERT_SQL, record)
        conn.execute(
            '''INSERT INTO scrape_log(source_url, status, http_status, error, attempted_at)
               VALUES (?, 'ok', ?, NULL, ?)
               ON CONFLICT(source_url) DO UPDATE SET
                   status='ok', http_status=excluded.http_status,
                   error=NULL, attempted_at=excluded.attempted_at''',
            (source_url, http_status, attempted_at),
        )
    except Exception as error:
        conn.execute(
            '''INSERT INTO scrape_log(source_url, status, http_status, error, attempted_at)
               VALUES (?, 'error', ?, ?, ?)
               ON CONFLICT(source_url) DO UPDATE SET
                   status='error', http_status=excluded.http_status,
                   error=excluded.error, attempted_at=excluded.attempted_at''',
            (source_url, http_status, str(error)[:1000], attempted_at),
        )

    if index % 20 == 0 or index == len(pending_urls):
        conn.commit()
        saved = conn.execute("SELECT COUNT(*) FROM songs").fetchone()[0]
        failures = conn.execute(
            "SELECT COUNT(*) FROM scrape_log WHERE status='error'"
        ).fetchone()[0]
        print(f"Fetched {index:,}/{len(pending_urls):,}; saved {saved:,}; errors {failures:,}")

    time.sleep(REQUEST_DELAY_SECONDS)

conn.commit()
print("Crawl complete. Successful song rows:", conn.execute("SELECT COUNT(*) FROM songs").fetchone()[0])
print("Failed URLs are in scrape_log; rerun this cell to retry them.")

In [ ]:
# Quick catalogue summary
row_count = conn.execute("SELECT COUNT(*) FROM songs").fetchone()[0]
print(f"Songs saved: {row_count:,}")

for column in ["parjaay", "taal", "raag", "ritu"]:
    values = conn.execute(
        f"SELECT COUNT(DISTINCT {column}) FROM songs WHERE {column} IS NOT NULL"
    ).fetchone()[0]
    missing = conn.execute(
        f"SELECT COUNT(*) FROM songs WHERE {column} IS NULL OR trim({column}) = ''"
    ).fetchone()[0]
    print(f"{column}: {values} distinct values; {missing:,} songs have no value listed")

sample = conn.execute(
    '''SELECT title_roman, parjaay, taal, raag, ritu, written_on, collection, source_url
       FROM songs ORDER BY title_roman LIMIT 5'''
).fetchall()
for item in sample:
    print(dict(item))

## Ask factual questions from the database

This baseline supports song-title questions and simple category queries. It only reports values present on the source page; a blank field is reported as unlisted.

In [ ]:
FIELD_TERMS = {
    "taal": ["taal", "tal", "rhythm", "তাল"],
    "parjaay": ["parjaay", "parjay", "পর্যায়", "পর্যায়"],
    "raag": ["raag", "raga", "রাগ"],
    "ritu": ["ritu", "rhitu", "season", "ঋতু"],
    "written_on": ["written on", "written", "year", "when was"],
    "collection": ["collection", "book"],
    "swarabitan": ["swarabitan"],
    "notation_by": ["notation by", "notator"],
}

STOPWORDS = set(
    "what is the of for in this that song songs tell me about which find list "
    "show give use uses using are was were by and".split()
)


def normalize(value):
    value = (value or "").lower()
    return re.sub(r"[^a-z0-9\u0980-\u09ff]+", " ", value).strip()


def detect_field(question):
    lower = question.lower()
    for field, terms in FIELD_TERMS.items():
        if any(term in lower for term in terms):
            return field
    return None


def identify_song(question, field):
    q = normalize(question)
    remove_terms = set(STOPWORDS)
    if field:
        for term in FIELD_TERMS[field]:
            remove_terms.update(normalize(term).split())
    q_words = [word for word in q.split() if word not in remove_terms]
    q_clean = " ".join(q_words)
    best_row, best_score = None, 0.0

    for row in conn.execute("SELECT * FROM songs WHERE scrape_status='ok'"):
        title = normalize(row["title_roman"])
        if not title:
            continue
        if title in q:
            score = 1.0
        elif q_clean and title in q_clean:
            score = 1.0
        else:
            ratio = SequenceMatcher(None, title, q_clean).ratio() if q_clean else 0.0
            title_words = set(title.split())
            overlap = len(title_words & set(q_words)) / max(1, len(title_words))
            score = max(ratio, overlap)
        if score > best_score:
            best_row, best_score = row, score
    return best_row, best_score, q_clean


def answer_question(question):
    field = detect_field(question)
    row, score, q_clean = identify_song(question, field)

    if row is not None and score >= 0.58:
        if field:
            value = row[field]
            answer = value if value else "not listed on this song page"
            label = field.replace("_", " ")
            return f"{row['title_roman']}: {label} is {answer}. Source: {row['source_url']}"

        details = [
            f"{name}: {row[name]}"
            for name in ["parjaay", "taal", "raag", "ritu", "written_on", "collection"]
            if row[name]
        ]
        summary = "; ".join(details) if details else "No structured metadata was listed."
        return f"{row['title_roman']}: {summary}. Source: {row['source_url']}"

    if field and q_clean:
        value_term = q_clean.strip()
        rows = conn.execute(
            f"SELECT title_roman, {field}, source_url FROM songs "
            f"WHERE lower(COALESCE({field}, '')) LIKE ? ORDER BY title_roman LIMIT 20",
            (f"%{value_term.lower()}%",),
        ).fetchall()
        if rows:
            titles = ", ".join(item["title_roman"] for item in rows)
            return f"Songs with {field.replace('_', ' ')} matching {value_term}: {titles}. Sources: " + "; ".join(item["source_url"] for item in rows)

    return (
        "I couldn't match a song or listed metadata value. Try a Romanized song title, "
        "or ask for taal, parjaay, raag, ritu/season, year, collection, or notation credit."
    )


print(answer_question("What is the taal of Aami ekla cholechhi?"))
print(answer_question("Which songs use Dadra taal?"))

## Train a small character-level transformer

This section follows the supplied gpt_dev.py in spirit, but trains on the site's available transliteration instead of Tiny Shakespeare. It learns text continuation. Keep the SQLite lookup above as the source for exact song facts.

In [ ]:
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE)

song_rows = conn.execute(
    '''SELECT title_roman, transliteration FROM songs
       WHERE scrape_status='ok' AND transliteration IS NOT NULL AND trim(transliteration) != '' '''
).fetchall()
random.Random(42).shuffle(song_rows)
split_at = max(1, int(0.9 * len(song_rows)))
train_text = "\n\n".join(row["transliteration"] for row in song_rows[:split_at])
val_text = "\n\n".join(row["transliteration"] for row in song_rows[split_at:])

if len(train_text) < 10_000 or len(val_text) < 1_000:
    raise RuntimeError("Not enough transliteration text. Finish the scrape and check the pages before training.")

chars = sorted(set(train_text + val_text))
stoi = {character: index for index, character in enumerate(chars)}
itos = {index: character for character, index in stoi.items()}
encode = lambda text: [stoi[character] for character in text]
decode = lambda ids: "".join(itos[int(index)] for index in ids)

train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)
VOCAB_SIZE = len(chars)
BLOCK_SIZE = 128
BATCH_SIZE = 32
N_EMBD = 64
N_HEAD = 4
N_LAYER = 2
DROPOUT = 0.1


def get_batch(split):
    data = train_data if split == "train" else val_data
    starts = torch.randint(0, len(data) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([data[i:i + BLOCK_SIZE] for i in starts])
    y = torch.stack([data[i + 1:i + BLOCK_SIZE + 1] for i in starts])
    return x.to(DEVICE), y.to(DEVICE)


class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(N_EMBD, head_size, bias=False)
        self.query = nn.Linear(N_EMBD, head_size, bias=False)
        self.value = nn.Linear(N_EMBD, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        batch, time_steps, channels = x.shape
        k = self.key(x)
        q = self.query(x)
        weights = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        weights = weights.masked_fill(self.tril[:time_steps, :time_steps] == 0, float("-inf"))
        weights = self.dropout(F.softmax(weights, dim=-1))
        return weights @ self.value(x)


class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = N_EMBD // N_HEAD
        self.heads = nn.ModuleList([Head(head_size) for _ in range(N_HEAD)])
        self.proj = nn.Linear(N_EMBD, N_EMBD)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        return self.dropout(self.proj(torch.cat([head(x) for head in self.heads], dim=-1)))


class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_EMBD, 4 * N_EMBD),
            nn.GELU(),
            nn.Linear(4 * N_EMBD, N_EMBD),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = MultiHeadAttention()
        self.feed_forward = FeedForward()
        self.ln1 = nn.LayerNorm(N_EMBD)
        self.ln2 = nn.LayerNorm(N_EMBD)

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        return x + self.feed_forward(self.ln2(x))


class TagoreCharGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.position_embedding = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.final_norm = nn.LayerNorm(N_EMBD)
        self.language_head = nn.Linear(N_EMBD, VOCAB_SIZE)

    def forward(self, idx, targets=None):
        batch, time_steps = idx.shape
        positions = torch.arange(time_steps, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        logits = self.language_head(self.final_norm(self.blocks(x)))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=400, temperature=0.9):
        for _ in range(max_new_tokens):
            context = idx[:, -BLOCK_SIZE:]
            logits, _ = self(context)
            logits = logits[:, -1, :] / temperature
            next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx


@torch.no_grad()
def estimate_loss(model, eval_batches=20):
    result = {}
    model.eval()
    for split in ["train", "val"]:
        losses = []
        for _ in range(eval_batches):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses.append(loss.item())
        result[split] = sum(losses) / len(losses)
    model.train()
    return result


model = TagoreCharGPT().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
MAX_ITERS = 600 if DEVICE == "cuda" else 250
EVAL_INTERVAL = 100

for step in range(MAX_ITERS):
    if step % EVAL_INTERVAL == 0 or step == MAX_ITERS - 1:
        losses = estimate_loss(model)
        print(f"step {step}: train {losses['train']:.3f}; validation {losses['val']:.3f}")
    x, y = get_batch("train")
    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

start_char = "\n" if "\n" in stoi else chars[0]
context = torch.tensor([[stoi[start_char]]], dtype=torch.long, device=DEVICE)
print(decode(model.generate(context)[0].tolist()))

MODEL_PATH = "tagore_char_gpt.pt"
torch.save(
    {
        "state_dict": model.state_dict(),
        "stoi": stoi,
        "itos": itos,
        "vocab_size": VOCAB_SIZE,
        "block_size": BLOCK_SIZE,
        "n_embd": N_EMBD,
        "n_head": N_HEAD,
        "n_layer": N_LAYER,
    },
    MODEL_PATH,
)
print("Saved transformer weights:", MODEL_PATH)

## Save the database and model

The database and transformer checkpoint are saved in the current Colab runtime. Download them before the runtime is deleted.

In [ ]:
from google.colab import files

files.download(DB_PATH)
files.download(MODEL_PATH)